## LG Aimers 9기 — Tree + TabM 전체 학습 및 네 번째 제출 (Colab A100)

08 Forward CV에서 선택된 앙상블을 2019~2024 전체 데이터로 재현한다. 기존 LightGBM·CatBoost R/F 라우팅을 유지하고, R 행에만 Brier-loss TabM을 30% 확률 혼합한다. F 행은 `submit_03`의 tree route와 정확히 같게 유지한다.

### 0. Drive 마운트와 프로젝트 확인

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

if Path('/content').exists():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

candidates = [
    Path('/content/drive/MyDrive/LG_aimers_9th'),
    Path('/content/drive/MyDrive/LG_Aimers_9th'),
    Path('/content/drive/MyDrive/LG Aimers 9th'),
    Path.cwd(),
    Path.cwd().parent,
]
required_relative = [
    Path('data/train.csv'),
    Path('data/test.csv'),
    Path('data/sample_submission.csv'),
    Path('src/features.py'),
    Path('src/submission_entrypoint.py'),
    Path('experiments/04_train_submit.py'),
    Path('experiments/07c_train_submit.py'),
    Path('experiments/08_tabm_cv.py'),
    Path('experiments/08b_final_train_submit.py'),
    Path('outputs/f_routing_v1/latest_run.json'),
    Path('outputs/tabm_cv_v1/latest_run.json'),
]
PROJECT_ROOT = next(
    (path for path in candidates if all((path / rel).exists() for rel in required_relative)),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        '프로젝트 파일 또는 07b·08 outputs가 없습니다. required_relative을 확인하세요.'
    )
PROJECT_ROOT = PROJECT_ROOT.resolve()
print('PROJECT_ROOT:', PROJECT_ROOT)
print('train.csv size (GB):', round((PROJECT_ROOT / 'data/train.csv').stat().st_size / 1e9, 3))

### 1. 제출 런타임 설치와 A100 확인

LightGBM·CatBoost와 공식 `tabm==0.0.3`을 설치한다. Tree 학습과 TabM 학습 모두 A100을 사용한다.

In [ ]:
install = subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q',
        'lightgbm==4.6.0', 'catboost==1.2.8', 'tabm==0.0.3',
    ],
    capture_output=True,
    text=True,
)
if install.returncode != 0:
    print(install.stdout)
    print(install.stderr)
    raise RuntimeError('제출 모델 패키지 설치 실패')

import catboost
import lightgbm
import numpy as np
import pandas as pd
import tabm
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU가 없습니다. Colab 런타임을 A100으로 변경하세요.')
gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print('Python:', sys.version.split()[0])
print('LightGBM:', lightgbm.__version__, '| CatBoost:', catboost.__version__)
print('PyTorch:', torch.__version__, '| TabM:', tabm.__version__)
print('GPU:', gpu_name, '| memory (GB):', round(gpu_memory_gb, 1))
if 'A100' not in gpu_name.upper():
    raise RuntimeError('08b는 A100 기준으로 검증했습니다. 런타임 GPU를 A100으로 변경하세요.')

### 2. 07b Tree 경로와 08 TabM 선택 결과 재확인

In [ ]:
ROUTING_LATEST = PROJECT_ROOT / 'outputs/f_routing_v1/latest_run.json'
TABM_LATEST = PROJECT_ROOT / 'outputs/tabm_cv_v1/latest_run.json'
with ROUTING_LATEST.open(encoding='utf-8') as handle:
    routing_latest = json.load(handle)
with TABM_LATEST.open(encoding='utf-8') as handle:
    tabm_latest = json.load(handle)
routing = routing_latest['summary']
tabm_result = tabm_latest['summary']
checks = pd.DataFrame([
    ['07b signature', routing['signature']],
    ['07b selected route', routing['selected_route']],
    ['07b status', routing['selection_status']],
    ['08 signature', tabm_result['signature']],
    ['08 selected candidate', tabm_result['selected_candidate']],
    ['08 status', tabm_result['selection_status']],
    ['08 recent R Brier', tabm_result['selected_recent_r_brier']],
    ['08 recent R BSS', tabm_result['selected_recent_r_bss_raw']],
    ['08 recent R delta', tabm_result['selected_delta_recent_r_brier_vs_base']],
], columns=['check', 'value'])
display(checks)
assert routing['signature'] == '46f3d7d0000e'
assert routing['selected_route'] == 'current_r__latest_f1_f'
assert tabm_result['signature'] == 'a960bb6f17e7'
assert tabm_result['selected_candidate'] == 'base_tabm_brier_w30'

### 3. 최종 앙상블 레시피 고정

In [ ]:
recipe = pd.DataFrame([
    ['LightGBM', 'R 2019~2024, half-life 2.5, pitchmix_shape'],
    ['CatBoost base', 'R/F 2019~2024, half-life 0.5, all_new'],
    ['CatBoost F route', '전체 R + 2024 F, half-life 0.5, all_new'],
    ['Tree base', 'LightGBM 65% + routed CatBoost 35% logit blend'],
    ['TabM', 'R 2019~2024, Brier loss, half-life 1.5, 1 epoch'],
    ['최종 R', 'Tree base 70% + TabM 30% probability blend'],
    ['최종 F', 'Tree base 100%; TabM 미적용'],
    ['test 사용', '현재 행의 원시 피처만 사용; test 집계·보정 없음'],
], columns=['component', 'fixed recipe'])
display(recipe)

### 4. 입력 데이터 사전 점검

In [ ]:
train_check = pd.read_csv(
    PROJECT_ROOT / 'data/train.csv',
    usecols=['row_id', 'season', 'game_type', 'control_success'],
)
test_check = pd.read_csv(
    PROJECT_ROOT / 'data/test.csv',
    usecols=['row_id', 'season', 'game_type'],
)
sample_check = pd.read_csv(PROJECT_ROOT / 'data/sample_submission.csv')
assert train_check['row_id'].notna().all() and train_check['row_id'].is_unique
assert test_check['row_id'].notna().all() and test_check['row_id'].is_unique
assert set(train_check['control_success'].unique()) <= {0, 1}
assert sorted(train_check['season'].unique().tolist()) == [2019, 2020, 2021, 2022, 2023, 2024]
assert set(test_check['row_id']) == set(sample_check['row_id'])
display(train_check.groupby(['season', 'game_type'], observed=True)['control_success'].agg(['size', 'mean']))
display(test_check.groupby(['season', 'game_type'], observed=True).size().rename('local_sample_rows'))
print('주의: 배포 test.csv는 형식 확인용이며, 이 통계를 학습 설정에 사용하지 않습니다.')

### 5. 전체 학습 및 `submit_04.zip` 생성

세 tree 모델과 TabM을 순차 학습한 뒤 native 저장·재로딩·행 독립성·F 고정·별도 임시 폴더 end-to-end 실행까지 자동 검사한다. 마지막의 `All checks passed.`가 나오기 전에는 제출하지 않는다.

In [ ]:
THREADS = max(1, min(8, os.cpu_count() or 2))
OUTPUT_DIR = PROJECT_ROOT / 'outputs/final_submission_v4'
command = [
    sys.executable,
    str(PROJECT_ROOT / 'experiments/08b_final_train_submit.py'),
    '--train-path', str(PROJECT_ROOT / 'data/train.csv'),
    '--data-dir', str(PROJECT_ROOT / 'data'),
    '--output-dir', str(OUTPUT_DIR),
    '--routing-latest-path', str(ROUTING_LATEST),
    '--tabm-latest-path', str(TABM_LATEST),
    '--threads', str(THREADS),
    '--lgbm-estimators', '500',
    '--catboost-iterations', '85',
    '--catboost-task-type', 'GPU',
    '--tabm-device', 'cuda',
    '--tabm-batch-size', '4096',
    '--tabm-epochs', '1',
    '--num-workers', '0',
]
print('Running:', ' '.join(command))
started = time.perf_counter()
subprocess.run(command, check=True)
print('elapsed minutes:', round((time.perf_counter() - started) / 60, 1))

### 6. 전체 학습 결과와 모델 구성 확인

In [ ]:
SUMMARY_PATH = OUTPUT_DIR / 'training_summary.json'
if not SUMMARY_PATH.exists():
    raise FileNotFoundError(SUMMARY_PATH)
with SUMMARY_PATH.open(encoding='utf-8') as handle:
    summary = json.load(handle)
training_rows = []
for key in ['lgbm', 'catboost_base', 'catboost_f_route', 'tabm']:
    model = summary[key]
    training_rows.append({
        'model': key,
        'rows': model['train_rows'],
        'target/prior': model.get('weighted_target_prior', model.get('target_rate')),
        'features': model.get('feature_count', model.get('numeric_feature_count')),
        'seconds': model['train_seconds'],
    })
display(pd.DataFrame(training_rows))
display(pd.DataFrame([
    ['selected neural candidate', summary['selected_neural_candidate']],
    ['Tree R/F route', summary['selected_route']],
    ['R tree weight', summary['neural_blend']['tree_weight']],
    ['R TabM weight', summary['neural_blend']['tabm_weight']],
    ['F policy', summary['neural_blend']['F_policy']],
], columns=['item', 'value']))

### 7. 직렬화·전처리·앙상블·행 독립성 감사

In [ ]:
audit = summary['audit']
tabm_parity = summary['tabm_final_inference_feature_parity']
checks = pd.DataFrame([
    ['audit status', audit['status']],
    ['TabM feature parity', tabm_parity['status']],
    ['TabM numeric max diff', tabm_parity['numeric_max_abs_difference']],
    ['TabM categorical exact', tabm_parity['categorical_exact']],
    ['final composition max diff', audit['final_composition_max_abs_difference']],
    ['F vs submit03 base max diff', audit['F_vs_schema3_base_max_abs_difference']],
    ['shuffle max diff', audit['row_independence_max_abs_difference']['shuffle']],
    ['chunk max diff', audit['row_independence_max_abs_difference']['chunk']],
    ['single-row max diff', audit['row_independence_max_abs_difference']['single_row']],
    ['end-to-end rows', summary['end_to_end']['rows']],
], columns=['check', 'value'])
display(checks)
assert audit['status'] == 'passed'
assert audit['F_predictions_exact_schema3_base']
assert tabm_parity['status'] == 'passed'
assert summary['end_to_end']['returncode'] == 0

### 8. 모델 파일과 제출 ZIP 구조 확인

최상위에는 `model/`, `script.py`, `requirements.txt`만 있어야 한다. `data/`, `output/`, 노트북, 소스 폴더와 `__pycache__`는 포함하지 않는다.

In [ ]:
SUBMIT_ZIP = Path(summary['package']['zip_path'])
expected = {
    'model/lgbm_model.txt',
    'model/catboost_base.cbm',
    'model/catboost_f_route.cbm',
    'model/tabm_state.pt',
    'model/artifacts.joblib',
    'script.py',
    'requirements.txt',
}
with zipfile.ZipFile(SUBMIT_ZIP) as archive:
    names = set(archive.namelist())
    members = pd.DataFrame([
        {
            'name': info.filename,
            'bytes': info.file_size,
            'compressed_bytes': info.compress_size,
        }
        for info in archive.infolist()
    ])
    assert names == expected
    assert archive.testzip() is None
assert SUBMIT_ZIP.name == 'submit_04.zip'
assert len(SUBMIT_ZIP.name) <= 30
display(members)
print('제출 파일:', SUBMIT_ZIP)
print('ZIP size (MB):', round(SUBMIT_ZIP.stat().st_size / 1e6, 2))
print('SHA-256:', summary['package']['zip_sha256'])

### 9. 제출 전 최종 판정

In [ ]:
validation = summary['validation_reference']
decision = pd.DataFrame([
    ['08 signature', validation['source_08_signature']],
    ['selected candidate', validation['candidate']],
    ['recent R Brier', validation['recent_r_brier']],
    ['recent R BSS', validation['recent_r_bss_raw']],
    ['recent R delta', validation['delta_recent_r_brier_vs_base']],
    ['worst fold delta', validation['worst_fold_delta']],
    ['package audit', audit['status']],
    ['filename length', len(SUBMIT_ZIP.name)],
], columns=['item', 'value'])
display(decision)
print('판정: 모든 검사가 통과했으면 submit_04.zip 자체를 압축 해제하지 않고 DACON에 제출합니다.')

### 10. 검토용 결과 ZIP 생성

실행 후 `submit_04.zip`, `submit04_review.zip`, 실행 결과가 저장된 08b 노트북을 전달한다.

In [ ]:
review_files = [
    OUTPUT_DIR / 'training_summary.json',
    OUTPUT_DIR / 'lgbm_feature_importance.csv',
    OUTPUT_DIR / 'catboost_base_feature_importance.csv',
    OUTPUT_DIR / 'catboost_f_route_feature_importance.csv',
]
REVIEW_ZIP = OUTPUT_DIR / 'submit04_review.zip'
with zipfile.ZipFile(REVIEW_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in review_files:
        if not path.exists():
            raise FileNotFoundError(path)
        archive.write(path, arcname=path.name)
print('제출 ZIP:', SUBMIT_ZIP)
print('검토 ZIP:', REVIEW_ZIP)
print('검토 ZIP 크기 (MB):', round(REVIEW_ZIP.stat().st_size / 1e6, 3))
print('함께 전달: submit_04.zip + submit04_review.zip + 실행된 08b_final_train_submit.ipynb')